# Importing libraries and loading the cleaned data

In [1]:
import pandas as pd
df = pd.read_csv('cleaned_online_retail.csv', parse_dates=['InvoiceDate'])
df.shape

(779425, 9)

# Set the snapshot date

In [2]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(snapshot_date)

2011-12-10 12:50:00


This is our reference point for calculating "Recency"

# Calculate Recency, Frequency, and Monetary per customer

In [3]:
rfm = df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

rfm.shape

(5878, 4)

# Look at the actual values

In [4]:
rfm.head(10)


,Customer ID,Recency,Frequency,Monetary
0,12346,326,12,77556.46
1,12347,2,8,4921.53
2,12348,75,5,2019.40
3,12349,19,4,4428.69
4,12350,310,1,334.40
5,12351,375,1,300.93
6,12352,36,10,2849.84
7,12353,204,2,406.76
8,12354,232,1,1079.40
9,12355,214,2,947.61


In [5]:
rfm.describe()

,Customer ID,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,201.331916,6.289384,2955.904095
std,1715.572666,209.338707,13.009406,14440.852688
min,12346.000000,1.000000,1.000000,2.950000
25%,13833.250000,26.000000,1.000000,342.280000
50%,15314.500000,96.000000,3.000000,867.740000
75%,16797.750000,380.000000,7.000000,2248.305000
max,18287.000000,739.000000,398.000000,580987.040000


# Score each metric 1–5 using quintiles

In [6]:
# Recency: lower is better, so 5 = most recent
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frequency: higher is better. We use .rank(method='first') to break ties
# since many customers share the same frequency (e.g. lots of people with exactly 1 order)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

# Monetary: higher is better
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm.head(10)

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,12346,326,12,77556.46,2,5,5
1,12347,2,8,4921.53,5,4,5
2,12348,75,5,2019.40,3,4,4
3,12349,19,4,4428.69,5,3,5
4,12350,310,1,334.40,2,1,2
5,12351,375,1,300.93,2,1,2
6,12352,36,10,2849.84,4,5,4
7,12353,204,2,406.76,2,2,2
8,12354,232,1,1079.40,2,1,3
9,12355,214,2,947.61,2,2,3


# Combine scores and build segment labels

In [7]:
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Sum'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

rfm.head(10)

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Sum
0,12346,326,12,77556.46,2,5,5,255,12
1,12347,2,8,4921.53,5,4,5,545,14
2,12348,75,5,2019.40,3,4,4,344,11
3,12349,19,4,4428.69,5,3,5,535,13
4,12350,310,1,334.40,2,1,2,212,5
5,12351,375,1,300.93,2,1,2,212,5
6,12352,36,10,2849.84,4,5,4,454,13
7,12353,204,2,406.76,2,2,2,222,6
8,12354,232,1,1079.40,2,1,3,213,6
9,12355,214,2,947.61,2,2,3,223,7


This gives each customer a combined code e.g., "255" and a simple total score e.g., 2+5+5=12 this is useful for quick sorting,however, the real segmentation logic uses R and F specifically, not just the sum.

# Map customers into named segments

In [8]:
def segment_customer(row):
    r, f = row['R_Score'], row['F_Score']
    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r == 3 and f <= 2:
        return 'Promising'
    elif r <= 2 and f >= 4:
        return 'At Risk'
    elif r <= 2 and f == 3:
        return 'Needs Attention'
    elif r <= 2 and f <= 2 and row['M_Score'] >= 4:
        return 'Cannot Lose Them'
    elif r == 1 and f <= 2:
        return 'Lost'
    else:
        return 'Hibernating'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

rfm['Segment'].value_counts()

Segment
Champions           1482
Loyal Customers     1221
Lost                 868
Hibernating          592
Needs Attention      471
New Customers        443
Promising            385
At Risk              353
Cannot Lose Them      63
Name: count, dtype: int64

# Evaluating the business impact by calculating revenue by segment

In [9]:
seg_summary = rfm.groupby('Segment').agg(
    Customers=('Customer ID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean'),
    Total_Revenue=('Monetary', 'sum')
).sort_values('Total_Revenue', ascending=False).round(1)

seg_summary

,Customers,Avg_Recency,Avg_Frequency,Avg_Monetary,Total_Revenue
Segment,,,,,
Champions,1482,20.4,15.7,8113.6,12024330.1
Loyal Customers,1221,78.6,5.4,2055.7,2510046.3
At Risk,353,344.2,7.6,3089.8,1090694.3
Needs Attention,471,387.8,3.0,1058.8,498690.0
New Customers,443,28.1,1.5,885.5,392267.0
Lost,868,557.9,1.2,307.3,266765.0
Hibernating,592,318.6,1.3,351.6,208174.5
Promising,385,107.1,1.4,530.8,204349.7
Cannot Lose Them,63,422.5,1.6,2849.0,179487.3


# Save your RFM results

In [10]:
rfm.to_csv('rfm_segments.csv', index=False)
print("Saved!")

Saved!
